<a href="https://colab.research.google.com/github/Rajfekar/PythonML/blob/main/Stable_Diffusion_Flowers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install diffusers transformers accelerate datasets torchvision --quiet
!pip install ftfy scipy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

dataset = load_dataset("nelorth/oxford-flowers")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-12de94e121bdbe(…):   0%|          | 0.00/303M [00:00<?, ?B/s]

data/test-00000-of-00001-96eeec628415add(…):   0%|          | 0.00/43.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7169 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1020 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 7169
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1020
    })
})


In [16]:
from torchvision import transforms

image_size = 256

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

def transform_examples(example):
    # Corrected: example["image"] is expected to be a list of PIL Image objects
    example["pixel_values"] = [transform(img.convert("RGB")) for img in example["image"]]
    # Remove the original 'image' column to prevent DataLoader from trying to collate it
    del example["image"]
    return example

dataset = dataset.with_transform(transform_examples)

In [17]:
from transformers import CLIPTokenizer, CLIPTextModel

tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.pre_layrnorm.bias                                 | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.embeddings.position_ids 

In [18]:
from diffusers import UNet2DConditionModel

unet = UNet2DConditionModel(
    sample_size=256,
    in_channels=3,
    out_channels=3,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 512),
    down_block_types=(
        "DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "DownBlock2D"
    ),
    up_block_types=(
        "UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"
    ),
    cross_attention_dim=512,
)

In [19]:
from diffusers import DDPMScheduler

noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

In [21]:
import torch
from torch.utils.data import DataLoader

# Check for CUDA availability and set device accordingly
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

unet.to(device)
text_encoder.to(device)

optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-4)

dataloader = DataLoader(dataset["train"], batch_size=8, shuffle=True)

epochs = 5

for epoch in range(epochs):
    for batch in dataloader:

        images = batch["pixel_values"].to(device)

        # Example prompt (flower class)
        prompts = ["a flower"] * images.shape[0]

        inputs = tokenizer(prompts, padding="max_length", return_tensors="pt").to(device)
        text_embeddings = text_encoder(**inputs).last_hidden_state

        # Sample noise
        noise = torch.randn_like(images)

        timesteps = torch.randint(0, noise_scheduler.num_train_timesteps, (images.shape[0],), device=device)

        noisy_images = noise_scheduler.add_noise(images, noise, timesteps)

        # Predict noise
        noise_pred = unet(noisy_images, timesteps, encoder_hidden_states=text_embeddings).sample

        loss = torch.nn.functional.mse_loss(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} Loss: {loss.item()}")

Using device: cpu


/usr/local/lib/python3.12/dist-packages/diffusers/configuration_utils.py:173: FutureWarning: Accessing config attribute `num_train_timesteps` directly via 'DDPMScheduler' object attribute is deprecated. Please access 'num_train_timesteps' over 'DDPMScheduler's config object instead, e.g. 'scheduler.config.num_train_timesteps'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)


KeyboardInterrupt: 

In [ ]:
from diffusers import DDPMPipeline

pipeline = DDPMPipeline(unet=unet, scheduler=noise_scheduler)
pipeline.to(device)

In [ ]:
import torch

prompt = "a red rose flower"

inputs = tokenizer([prompt], return_tensors="pt").to(device)
text_embeddings = text_encoder(**inputs).last_hidden_state

# Start from noise
image = torch.randn((1, 3, 256, 256)).to(device)

for t in reversed(range(noise_scheduler.num_train_timesteps)):
    timestep = torch.tensor([t]).to(device)

    noise_pred = unet(image, timestep, encoder_hidden_states=text_embeddings).sample

    image = noise_scheduler.step(noise_pred, timestep, image).prev_sample

# Convert to image
image = (image / 2 + 0.5).clamp(0, 1)
image = image.cpu().permute(0, 2, 3, 1).numpy()[0]

import matplotlib.pyplot as plt
plt.imshow(image)
plt.axis("off")